In [14]:
import pandas as pd
import numpy as np
import pickle
import re

from sklearn.feature_extraction.text import TfidfVectorizer

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download("punkt")
nltk.download("stopwords")
nltk.download("wordnet")

[nltk_data] Downloading package punkt to C:\Users\Abarna
[nltk_data]     Studio\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Abarna
[nltk_data]     Studio\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to C:\Users\Abarna
[nltk_data]     Studio\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

## load dataset

In [15]:
faq = pd.read_csv("../data/customer_faq_500.csv")
faq.head()

,question,answer,category
0,Can I track multiple orders?,"Yes, each order has its own tracking informati...",Order Tracking
1,Why hasn't my order shipped?,Orders are usually processed within 24 hours. ...,Order Tracking
2,Where is my order?,You can track your order from the 'My Orders' ...,Order Tracking
3,Can I track multiple orders?,"Yes, each order has its own tracking informati...",Order Tracking
4,Why hasn't my order shipped?,Orders are usually processed within 24 hours. ...,Order Tracking


## Dataset info

In [16]:
print("Dataset Shape:", faq.shape)

print("\nColumns:")
print(faq.columns.tolist())

print("\nCategories:")
print(faq["category"].unique())

Dataset Shape: (500, 3)

Columns:
['question', 'answer', 'category']

Categories:
['Order Tracking' 'Returns' 'Refunds' 'Shipping' 'Payments'
 'Technical Support' 'Warranty' 'Account' 'Membership'
 'Product Information']


## NLP Text cleaning

In [17]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words("english"))

def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z ]", " ", text)
    words = text.split()
    words = [
        lemmatizer.lemmatize(word)
        for word in words
        if word not in stop_words
    ]
    return " ".join(words)

## Clean questions

In [18]:
faq["clean_question"] = faq["question"].apply(clean_text)
faq[["question","clean_question"]].head()

,question,clean_question
0,Can I track multiple orders?,track multiple order
1,Why hasn't my order shipped?,order shipped
2,Where is my order?,order
3,Can I track multiple orders?,track multiple order
4,Why hasn't my order shipped?,order shipped


## TF-IDF

In [19]:
vectorizer = TfidfVectorizer(max_features=300)
tfidf = vectorizer.fit_transform(faq["clean_question"])
keywords = vectorizer.get_feature_names_out()
print("Total Keywords:", len(keywords))
keywords[:50]

Total Keywords: 86


array(['accepted', 'accessory', 'accidental', 'account', 'accurate',
       'address', 'assistance', 'availability', 'benefit', 'cancel',
       'cash', 'change', 'check', 'claim', 'compare', 'contact',
       'covered', 'damage', 'delayed', 'delete', 'delivery', 'detail',
       'device', 'dimension', 'earn', 'email', 'express', 'fail', 'find',
       'free', 'get', 'gold', 'include', 'inspected', 'internationally',
       'issue', 'item', 'join', 'know', 'locked', 'long', 'mean',
       'member', 'membership', 'method', 'multiple', 'offer', 'online',
       'order', 'package'], dtype=object)

## Category keywords

In [20]:
category_keywords = {}
for category in faq["category"].unique():
    subset = faq[faq["category"] == category]
    vectorizer = TfidfVectorizer(max_features=15)
    X = vectorizer.fit_transform(subset["clean_question"])
    category_keywords[category] = vectorizer.get_feature_names_out().tolist()

In [22]:
for cat, words in category_keywords.items():
    print(cat)
    print(words)
    print("-"*50)

Order Tracking
['mean', 'multiple', 'order', 'shipment', 'shipped', 'track', 'transit']
--------------------------------------------------
Returns
['free', 'inspected', 'item', 'long', 'product', 'return', 'shipping', 'used']
--------------------------------------------------
Refunds
['cash', 'check', 'delayed', 'get', 'receive', 'refund', 'sent', 'status']
--------------------------------------------------
Shipping
['address', 'change', 'delayed', 'delivery', 'express', 'internationally', 'long', 'offer', 'package', 'ship', 'shipping', 'take']
--------------------------------------------------
Payments
['accepted', 'fail', 'method', 'online', 'pay', 'payment', 'secure', 'split', 'upi']
--------------------------------------------------
Technical Support
['assistance', 'contact', 'device', 'find', 'get', 'issue', 'remote', 'report', 'step', 'support', 'technical', 'troubleshooting', 'turn']
--------------------------------------------------
Warranty
['accidental', 'claim', 'covered', '

## Domain Synonyms

In [23]:
extra_synonyms = {
    "Refunds":[
        "refund",
        "money back",
        "reimbursement"
    ],
    "Returns":[
        "return",
        "replacement",
        "exchange"
    ],
    "Order Tracking":[
        "track",
        "tracking",
        "shipment",
        "delivery"
    ],
    "Payments":[
        "payment",
        "upi",
        "credit card",
        "wallet"
    ],
    "Shipping":[
        "shipping",
        "dispatch",
        "courier"
    ],
    "Account":[
        "login",
        "password",
        "profile"
    ],
    "Membership":[
        "membership",
        "subscription",
        "premium"
    ],
    "Warranty":[
        "repair",
        "guarantee"
    ],
    "Technical Support":[
        "technical",
        "issue",
        "bug",
        "support"
    ],
    "Product Information":[
        "product",
        "feature",
        "manual",
        "specification"
    ]
}

In [24]:
for category in category_keywords:

    if category in extra_synonyms:

        category_keywords[category].extend(
            extra_synonyms[category]
        )

    category_keywords[category] = list(
        set(category_keywords[category])
    )

In [25]:
question_examples = {}

for category in faq["category"].unique():

    question_examples[category] = faq[
        faq["category"] == category
    ]["question"].tolist()

In [26]:
nlp_pipeline = {

    "categories": faq["category"].unique().tolist(),

    "category_keywords": category_keywords,

    "question_examples": question_examples

}

In [27]:
with open("../data/nlp_pipeline.pkl","wb") as f:

    pickle.dump(nlp_pipeline,f)

print("NLP Pipeline Saved Successfully")

NLP Pipeline Saved Successfully


In [28]:
with open("../data/nlp_pipeline.pkl","rb") as f:

    nlp = pickle.load(f)

print(nlp.keys())

dict_keys(['categories', 'category_keywords', 'question_examples'])
